# Evaluation

This notebook will evaluate the performance of the retrival/reranking.

After making changes to the code you can assess the performance of the retrieval, reranking or LLM steps by running this notebook

In [17]:
import sys
from pathlib import Path

proj_dir = Path.cwd().parent.resolve()
sys.path.append(str(proj_dir))

## Retrieval
In this section we can load the results for different runs of `scripts/eval_retrieval.py` and compare the results

In [24]:
from pathlib import Path
import json
import pandas as pd

EVAL_RUNS_DIR = Path("../data/eval_runs")

def _resolve_eval_file(run_id_or_path) -> Path:
    p = Path(run_id_or_path)
    if p.exists():
        if p.is_dir():
            p = p / "retrieval_eval.json"
        return p
    rid = str(run_id_or_path).replace("run_", "")
    f = EVAL_RUNS_DIR / f"run_{rid}" / "retrieval_eval.json"
    if not f.exists():
        raise FileNotFoundError(f"Could not find retrieval_eval.json for run '{run_id_or_path}' in {EVAL_RUNS_DIR}")
    return f

def find_runs():
    if not EVAL_RUNS_DIR.exists():
        return []
    out = []
    for run_dir in sorted(EVAL_RUNS_DIR.glob("run_*")):
        f = run_dir / "retrieval_eval.json"
        if f.exists():
            out.append({"run_id": run_dir.name.replace("run_", ""), "path": f})
    return out

def _read_summary(run_id_or_path):
    f = _resolve_eval_file(run_id_or_path)
    data = json.loads(f.read_text())
    s = data.get("summary", {}) or {}
    pre = s.get("pre_rerank") or {}
    post = s.get("post_rerank") or {}
    return f, pre, post

# Long/stacked: one row per run per stage
def build_summary_table_long(run_ids_or_paths) -> pd.DataFrame:
    rows = []
    for r in run_ids_or_paths:
        f, pre, post = _read_summary(r)
        run_id = f.parent.name.replace("run_", "")
        for stage_name, m in [("pre_rerank", pre), ("post_rerank", post)]:
            if not m:
                continue
            rows.append({
                "run_id": run_id,
                "stage": stage_name,
                "k": m.get("k"),
                "alpha": m.get("alpha"),
                "reranker": m.get("use_reranker"),
                "judge_model": m.get("judge_model"),
                "n": m.get("n"),
                "total_records": m.get("total_records"),
                "filtered_unanswerable": m.get("filtered_unanswerable"),
                "hit@k": m.get("hit@k"),
                "mrr@k": m.get("mrr@k"),
                "ndcg@k": m.get("ndcg@k"),
                "precision@k": m.get("precision@k"),
                "recall@k": m.get("recall@k"),
                "path": str(f),
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values(["run_id", "stage"])

def style_summary(df: pd.DataFrame):
    df = df.copy()
    num_cols = [c for c in df.columns if any(key in c for key in ["alpha","hit@k","mrr@k","ndcg@k","precision@k","recall@k"])]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return (
        df.style
          .format({c: "{:.3f}" for c in num_cols})
          .set_table_styles([{"selector":"th","props":[("text-align","left")]}])
          .set_properties(**{"text-align": "right"}, subset=num_cols)
    )

# Examples:
runs = find_runs(); runs
ids = [r["run_id"] for r in runs][-3:]
style_summary(build_summary_table_long(ids))  # stacked view (separate rows)

,run_id,stage,k,alpha,reranker,judge_model,n,total_records,filtered_unanswerable,hit@k,mrr@k,ndcg@k,precision@k,recall@k,path
1,20250811_063338,post_rerank,3,0.500,True,None,122,172,50,0.811,0.673,0.709,0.270,0.811,../data/eval_runs/run_20250811_063338/retrieval_eval.json
0,20250811_063338,pre_rerank,10,0.500,True,None,122,172,50,0.869,0.685,0.730,0.087,0.869,../data/eval_runs/run_20250811_063338/retrieval_eval.json
3,20250811_064605,post_rerank,3,0.100,True,None,122,172,50,0.811,0.678,0.712,0.270,0.811,../data/eval_runs/run_20250811_064605/retrieval_eval.json
2,20250811_064605,pre_rerank,10,0.100,True,None,122,172,50,0.836,0.632,0.683,0.084,0.836,../data/eval_runs/run_20250811_064605/retrieval_eval.json
5,20250811_065224,post_rerank,3,0.900,True,None,122,172,50,0.779,0.673,0.701,0.260,0.779,../data/eval_runs/run_20250811_065224/retrieval_eval.json
4,20250811_065224,pre_rerank,10,0.900,True,None,122,172,50,0.861,0.696,0.737,0.086,0.861,../data/eval_runs/run_20250811_065224/retrieval_eval.json


An Alpha of 0.5 performaned better than 0.1 and 0.9, indicating that a hybrid approach is best for this system.

## LLM
Here we can compare the results of different runs which evaluate the performance of the LLM answering the questions using `scripts/eval_llm.py`

In [21]:
LLM_EVAL_RUNS_DIR = Path("../data/eval_runs")

def find_llm_runs():
    """List available LLM eval runs under ../data/eval_runs."""
    if not LLM_EVAL_RUNS_DIR.exists():
        return []
    runs = []
    for run_dir in sorted(LLM_EVAL_RUNS_DIR.glob("run_*")):
        f = run_dir / "llm_eval.json"
        if f.exists():
            runs.append({"run_id": run_dir.name.replace("run_", ""), "path": f})
    return runs

def _resolve_llm_eval_file(run_id_or_path) -> Path:
    """Resolve a run ID, run folder, or JSON file to the llm_eval.json path."""
    p = Path(run_id_or_path)
    if p.exists():
        if p.is_dir():
            p = p / "llm_eval.json"
        return p
    rid = str(run_id_or_path).replace("run_", "")
    f = LLM_EVAL_RUNS_DIR / f"run_{rid}" / "llm_eval.json"
    if not f.exists():
        raise FileNotFoundError(f"Could not find llm_eval.json for run '{run_id_or_path}' in {LLM_EVAL_RUNS_DIR}")
    return f

def load_llm_summary(run_id_or_path) -> dict:
    """Load top-level summary metrics for a run."""
    f = _resolve_llm_eval_file(run_id_or_path)
    data = json.loads(f.read_text())
    s = data.get("summary", {})
    return {
        "run_id": f.parent.name.replace("run_", ""),
        "n": s.get("n"),
        "avg_correctness": s.get("avg_correctness"),
        "avg_faithfulness": s.get("avg_faithfulness"),
        "avg_relevance": s.get("avg_relevance"),
        "pass_rate": s.get("pass_rate"),
        "path": str(f),
    }

def build_llm_summary_table(run_ids_or_paths) -> pd.DataFrame:
    rows = [load_llm_summary(r) for r in run_ids_or_paths]
    cols = ["run_id","n","avg_correctness","avg_faithfulness","avg_relevance","pass_rate","path"]
    df = pd.DataFrame(rows)
    return df[cols].sort_values("run_id")

def style_llm_summary(df: pd.DataFrame):
    df = df.copy()
    num_cols = [c for c in ["avg_correctness","avg_faithfulness","avg_relevance","pass_rate"] if c in df.columns]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return (
        df.style
          .format({c: "{:.3f}" for c in num_cols})
          .set_table_styles([{"selector":"th","props":[("text-align","left")]}])
          .set_properties(**{"text-align": "right"}, subset=num_cols)
    )

def load_llm_details(run_id_or_path) -> pd.DataFrame:
    """Per-question details for a run (id, type, correctness, faithfulness, relevance, verdict)."""
    f = _resolve_llm_eval_file(run_id_or_path)
    data = json.loads(f.read_text())
    return pd.DataFrame(data.get("details", []))

def build_llm_by_type_table(run_ids_or_paths) -> pd.DataFrame:
    """Flatten by_type summaries across runs into a table."""
    rows = []
    for r in run_ids_or_paths:
        f = _resolve_llm_eval_file(r)
        data = json.loads(f.read_text())
        s = data.get("summary", {})
        run_id = f.parent.name.replace("run_", "")
        by_type = s.get("by_type", {}) or {}
        for t, v in by_type.items():
            rows.append({
                "run_id": run_id,
                "type": t,
                "n": v.get("n"),
                "pass_rate": v.get("pass_rate"),
                "path": str(f),
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["pass_rate"] = pd.to_numeric(df["pass_rate"], errors="coerce")
    return df.sort_values(["run_id","type"])

def style_llm_by_type(df: pd.DataFrame):
    df = df.copy()
    if "pass_rate" in df.columns:
        df["pass_rate"] = pd.to_numeric(df["pass_rate"], errors="coerce")
    return (
        df.style
          .format({"pass_rate": "{:.3f}"})
          .set_table_styles([{"selector":"th","props":[("text-align","left")]}])
          .set_properties(**{"text-align": "right"}, subset=["pass_rate"])
    )

In [ ]:
runs = find_llm_runs(); runs
selected = [r["run_id"] for r in runs][-3:]
style_llm_summary(build_llm_summary_table(selected))

,run_id,n,avg_correctness,avg_faithfulness,avg_relevance,pass_rate,path
0,20250811_072834,75,0.960,0.984,0.980,0.947,../data/eval_runs/run_20250811_072834/llm_eval.json
